# Calculate maximum improvement or deterioration for all models

## Load libraries

In [1]:
import libs.tools as tools

## General setup

In [2]:
# Number of operators needed in bS=1 case
c_needed = 16

## Load simulation results

### Reference models

In [3]:
# Reference models, bI=1
base_bI1 = tools.prepare_statistics(c_needed, lambda rec: rec["mode"] == "fixed" and rec["b_min"] == 16 and isinstance(rec["bI"], int))
tools.load_statistics(base_bI1)

# Reference models, bI=1..4
base_bI1234 = tools.prepare_statistics(c_needed, lambda rec: rec["mode"] == "fixed" and rec["b_min"] == 16 and isinstance(rec["bI"], str))
tools.load_statistics(base_bI1234)

### Model 4 and 5

In [4]:
# Model 4, bI=1
model4_bI1 = tools.prepare_statistics(c_needed, lambda rec: rec["mode"] == "range" and rec["bI"] == 1)
tools.load_statistics(model4_bI1)

# Model 5, bI=1..4
model5_bI1234 = tools.prepare_statistics(c_needed, lambda rec: rec["mode"] == "range"  and isinstance(rec["bI"], str))
tools.load_statistics(model5_bI1234)

### Models 6 and 7

In [5]:
# Model 6, bI=1
model6_bI1 = tools.prepare_statistics(c_needed, lambda rec: rec["mode"] == "time-last" and rec["b_min"] == 16 and isinstance(rec["bI"], int))
tools.load_statistics(model6_bI1)

# Model 7, bI=1..4
model7_bI1234 = tools.prepare_statistics(c_needed, lambda rec: rec["mode"] == "time-last" and rec["b_min"] == 16 and isinstance(rec["bI"], str))
tools.load_statistics(model7_bI1234)

### Models 8 and 9

In [6]:
# Model 8, bI=1
model8_bI1 = tools.prepare_statistics(c_needed, lambda rec: rec["mode"] == "time-max" and rec["b_min"] == 16 and isinstance(rec["bI"], int))
tools.load_statistics(model8_bI1)

# Model 9, bI=1..4
model9_bI1234 = tools.prepare_statistics(c_needed, lambda rec: rec["mode"] == "time-max" and rec["b_min"] == 16 and isinstance(rec["bI"], str))
tools.load_statistics(model9_bI1234)

### Models 10 and 11

In [7]:
# Model 10, bI=1
model10_bI1 = tools.prepare_statistics(c_needed, lambda rec: rec["mode"] == "time-last" and rec["b_min"] == 12 and isinstance(rec["bI"], int))
tools.load_statistics(model10_bI1)

# Model 11, bI=1..4
model11_bI1234 = tools.prepare_statistics(c_needed, lambda rec: rec["mode"] == "time-last" and rec["b_min"] == 12 and isinstance(rec["bI"], str))
tools.load_statistics(model11_bI1234)

### Models 12 and 13

In [8]:
# Model 12, bI=1
model12_bI1 = tools.prepare_statistics(c_needed, lambda rec: rec["mode"] == "time-max" and rec["b_min"]==12 and isinstance(rec["bI"], int))
tools.load_statistics(model12_bI1)

# Model 13, bI=1..4
model13_bI1234 = tools.prepare_statistics(c_needed, lambda rec: rec["mode"] == "time-max" and rec["b_min"]==12 and isinstance(rec["bI"], str))
tools.load_statistics(model13_bI1234)

## Functions for calculating the results

In [9]:
def find_refence_model_EWrel(models: list[dict], model: dict, reference_condition) -> float:
    for test_model in models:
        if test_model["c"] != model["c"]:
            continue
        if test_model["model_rho"] != model["model_rho"]:
            continue
        if test_model["model_CV[S]"] != model["model_CV[S]"]:
            continue
        if reference_condition(model, test_model):
            return model["E[W]"] / test_model["E[W]"]
    raise ValueError("No reference found for " + model["name_short"])


def calc_change(models: list[dict], reference_models: list[dict], reference_condition) -> dict:
    changes = {"min_value": 1, "min_model": None, "min_name": "", "max_value": 1, "max_model": None, "max_name": ""}

    for model in models:
        change = find_refence_model_EWrel(reference_models, model, reference_condition)
        if change > changes["max_value"]:
            changes["max_value"] = change
            changes["max_model"] = model
            changes["max_name"] = model["name_short"]
        if change < changes["min_value"]:
            changes["min_value"] = change
            changes["min_model"] = model
            changes["min_name"] = model["name_short"]

    return changes

def calc_change_bI1(models: list[dict]) -> dict:
    global base_bI1
    return calc_change(models, base_bI1, lambda _, test_model: test_model["bI"] == 1) # type: ignore

def calc_change_bI1234(models: list[dict]) -> dict:
    global base_bI1234
    return calc_change(models, base_bI1234, lambda model, test_model: model["bI"] == test_model["bI"] ) # type: ignore


def print_change(changes: dict):
    print(f"Min\t{changes['min_value'] * 100:.2f}%\t{changes['min_name']}") # type: ignore
    print(f"Max\t{changes['max_value'] * 100:.2f}%\t{changes['max_name']}") # type: ignore

## Calculate results

In [10]:
print("Model 4:")
print_change(calc_change_bI1(model4_bI1))
print("Model 5:")
print_change(calc_change_bI1234(model5_bI1234))

print("Model 6:")
print_change(calc_change_bI1(model6_bI1))
print("Model 7:")
print_change(calc_change_bI1234(model7_bI1234))

print("Model 8:")
print_change(calc_change_bI1(model8_bI1))
print("Model 9:")
print_change(calc_change_bI1234(model9_bI1234))

print("Model 10:")
print_change(calc_change_bI1(model10_bI1))
print("Model 11:")
print_change(calc_change_bI1234(model11_bI1234))

print("Model 12:")
print_change(calc_change_bI1(model12_bI1))
print("Model 13:")
print_change(calc_change_bI1234(model11_bI1234))

Model 4:
Min	85.99%	S=Log, CV[S]=0.5, rho=60%, bI=1, bS=12..16
Max	107.98%	S=Log, CV[S]=1.5, rho=70%, bI=1, bS=12..16
Model 5:
Min	81.24%	S=Log, CV[S]=0.5, rho=60%, bI=1234, bS=12..16
Max	104.43%	S=Log, CV[S]=1.5, rho=80%, bI=4321, bS=12..16
Model 6:
Min	82.33%	S=Log, CV[S]=0.5, rho=60%, bI=1, bS=16, t=15
Max	136.56%	S=Log, CV[S]=1.5, rho=60%, bI=1, bS=16, t=15
Model 7:
Min	82.52%	S=Log, CV[S]=0.5, rho=60%, bI=1234, bS=16, t=45
Max	132.74%	S=Log, CV[S]=1.5, rho=60%, bI=4321, bS=16, t=15
Model 8:
Min	83.57%	S=Log, CV[S]=0.5, rho=60%, bI=1, bS=16, t=900
Max	113.41%	S=Log, CV[S]=1.5, rho=70%, bI=1, bS=16, t=900
Model 9:
Min	83.31%	S=Log, CV[S]=0.5, rho=60%, bI=2112, bS=16, t=900
Max	111.37%	S=Log, CV[S]=1.5, rho=70%, bI=4321, bS=16, t=900
Model 10:
Min	86.46%	S=Log, CV[S]=0.5, rho=60%, bI=1, bS=16, t=15
Max	107.66%	S=Log, CV[S]=1.5, rho=70%, bI=1, bS=16, t=15
Model 11:
Min	81.84%	S=Log, CV[S]=0.5, rho=60%, bI=1234, bS=16, t=15
Max	104.10%	S=Log, CV[S]=1.5, rho=75%, bI=4321, bS=16, t=15
Mo